# A Textbook-Style Exposition of Hybrid ALNS for the One-Dimensional Bin Packing Problem

---

## I. Formal Problem Statement

### I.1 Optimization problem and input model

The **one-dimensional bin packing problem (1D-BPP)** is defined as follows. We are given:
- a finite set of **items** indexed by $\mathcal{I} = \{1,2,\ldots,n\}$,
- strictly positive integer **sizes** $s_i \in \mathbb{Z}_{>0}$ for each $i \in \mathcal{I}$,
- identical **bins**, each of integer **capacity** $C \in \mathbb{Z}_{>0}$, with the assumption that $s_i \le C$ for all $i$ (otherwise no feasible solution exists).

A **feasible solution** is a partition of $\mathcal{I}$ into a collection of $m$ non-empty subsets (bins) $B_1, B_2, \ldots, B_m$ satisfying:

$$
\biguplus_{j=1}^{m} B_j = \mathcal{I},
\qquad
\sum_{i \in B_j} s_i \le C \quad \forall j \in \{1,\ldots,m\}.
$$

The **objective** is to minimize the number of bins used:

$$
\min_{B_1,\ldots,B_m} \; m.
$$

### I.2 Lower bound

A canonical **continuous relaxation lower bound** is:

$$
\mathrm{LB}_1 = \left\lceil \frac{\sum_{i=1}^{n} s_i}{C} \right\rceil.
$$

**Validity.** Any feasible packing places all item volume into bins of capacity $C$, so total used capacity is at least $\sum_i s_i$. Since each of the $m$ bins contributes at most $C$ units of capacity, we need $mC \ge \sum_i s_i$, i.e., $m \ge \sum_i s_i / C$. Because $m$ is integer, the ceiling follows. $\square$

The bound can be sharpened (e.g. via the $L_2$ bound of Martello and Toth, which accounts for large items that cannot coexist in a bin), but $\mathrm{LB}_1$ is sufficient as a progress indicator in this implementation.

### I.3 Computational complexity

1D-BPP is **strongly NP-hard** (by reduction from 3-Partition). This implies that, unless P = NP, no polynomial-time algorithm can solve all instances optimally. In practice:
- **Exact methods** (branch-and-bound, branch-and-price, column generation) are feasible for small-to-medium instances (up to a few hundred items) but scale poorly.
- **Approximation algorithms** (e.g. First-Fit Decreasing, Best-Fit Decreasing) run in $O(n \log n)$ and achieve a bounded approximation ratio: BFD is known to use at most $\frac{11}{9}\,\mathrm{OPT} + 6/9$ bins.
- **Metaheuristics** sacrifice optimality guarantees in exchange for scalable search over large instances, which is the focus of this solver.

## II. Constructive Initialization: First-Fit Decreasing and Best-Fit Decreasing

Before any metaheuristic search, a high-quality starting solution is needed. Both heuristics below sort items by non-increasing size, a strategy that is provably better than arbitrary ordering because large items are hardest to place and benefit most from seeing empty space.

**First-Fit Decreasing (FFD).** Process items in non-increasing size order. For each item, scan the open bins from first to last and place the item in the *first* bin with sufficient residual capacity. If no such bin exists, open a new one.

**Best-Fit Decreasing (BFD).** Same ordering, but instead of the first fitting bin, choose the bin that minimizes **residual capacity after placement** (i.e., the tightest fit). This greedy criterion reduces wasted space per step and typically produces fewer bins than FFD in practice.

The solver uses **FFD as the warm start** for ALNS. BFD is used separately as the **reference oracle** during offline training of the repair model (Section V), because BFD decisions encode strong packing intuitions that the model is trained to imitate.

## III. Metaheuristic Framework: Large-Neighborhood Search and its Adaptive Variant

### III.1 Why local search alone is insufficient

Classical local search improves a solution by applying small *moves* (e.g., relocating a single item). The **neighborhood** of a solution $x$ is the set of solutions reachable by one move. While efficient, small-neighborhood local search suffers from **local optima**: solutions from which no single move improves the objective, yet which are far from globally optimal. For 1D-BPP, local optima are dense and structured around specific item groupings.

### III.2 Large-Neighborhood Search (LNS)

LNS, introduced by Shaw (1998), addresses the local optima problem by operating on *large* implicit neighborhoods. Each iteration consists of two phases:

1. **Destroy**: partially deconstruct the current solution by removing a subset of items from their assigned bins. This produces a *partial solution* $\hat{x}$ and a set of **displaced items** $D$.
2. **Repair**: reinsert all displaced items into $\hat{x}$ to restore feasibility, producing a new complete solution $x'$.

The size of the neighborhood is implicitly exponential in $|D|$, because repair can produce any feasible completion of $\hat{x}$. This allows LNS to escape local optima that are unreachable by any sequence of small moves.

### III.3 Adaptive Large-Neighborhood Search (ALNS)

Ropke and Pisinger (2006) extended LNS by maintaining a **portfolio of destroy operators** and **adapting their selection probabilities** based on observed performance. The rationale is that no single destroy strategy dominates on all instances or at all stages of the search: random destruction explores broadly early on, while targeted destruction (e.g., removing poorly packed bins) is more effective near a local optimum.

In this solver, three destroy operators are available:

| Arm | Operator | Mechanism |
|-----|----------|-----------|
| 0 | **Random** | Randomly select $k$ bins and displace all their items. |
| 1 | **Worst-load** | Select the $k$ least-loaded bins (those with most wasted space) and displace their items. |
| 2 | **Related-item** | Select a seed item at random, then displace the $k$ items whose sizes are closest to the seed's size. |

The number of bins removed $k$ is sampled uniformly from $[k_{\min}, k_{\max}]$ with:

$$
k_{\min} = \max\!\left(1, \left\lfloor 0.05\,n \right\rfloor\right), \qquad k_{\max} = \max\!\left(k_{\min}+1, \left\lfloor 0.25\,n \right\rfloor\right),
$$

so between 5% and 25% of items are displaced per iteration.

Operator selection is governed by Thompson Sampling (Section V.1).

## IV. Acceptance Mechanism: Simulated Annealing

### IV.1 The acceptance criterion

Let $x$ be the **incumbent solution** and $x'$ the candidate produced by destroy-repair. Define the **cost difference**:

$$
\Delta = f(x') - f(x),
$$

where $f(\cdot)$ counts the number of non-empty bins. The **simulated annealing (SA) acceptance rule** is:

$$
\text{accept } x' \iff
\begin{cases}
\Delta \le 0, & \text{(always accept improvements or equal solutions)} \\
U < \exp\!\left(-\dfrac{\Delta}{T}\right), & \text{(accept degradation with probability } e^{-\Delta/T}\text{),}
\end{cases}
$$

where $U \sim \mathrm{Uniform}(0,1)$ and $T > 0$ is the **temperature**.

### IV.2 Interpretation of the acceptance probability

The term $e^{-\Delta/T}$ has two useful properties:

- **Monotone in $\Delta$**: worse moves ($\Delta$ large) are accepted with lower probability, so the search is not purely random.
- **Monotone in $T$**: at high $T$, $e^{-\Delta/T} \to 1$ and almost all moves are accepted (**exploration**); as $T \to 0$, $e^{-\Delta/T} \to 0$ and only improvements are accepted (**exploitation**).

This is the *Metropolis criterion* from statistical physics, where $T$ plays the role of thermodynamic temperature.

### IV.3 Geometric cooling schedule

The temperature is updated after every iteration:

$$
T_{t+1} = \alpha_{\text{cool}} \cdot T_t, \qquad \alpha_{\text{cool}} \in (0, 1).
$$

With $\alpha_{\text{cool}} = 0.9995$ (the solver default), temperature decays to roughly $e^{-1}$ of its starting value after $2000$ iterations, and to $e^{-2.5}$ after $5000$ iterations. The **initial temperature** $T_0 = 1/\ln 2 \approx 1.443$ is chosen so that at iteration 0 a move that worsens the solution by exactly one bin is accepted with probability $e^{-1/T_0} = 0.5$.

## V. Two Machine Learning Components

The solver incorporates two learning-based mechanisms that operate at different levels of the search. The first concerns *which destroy operator to invoke* at each iteration: rather than rotating through operators blindly, the solver maintains running estimates of how productive each one has been and uses those estimates to guide selection — adapting as the search progresses. The second concerns *how to reinsert displaced items* during repair: a model trained offline has learned to mimic the placement decisions of a strong constructive heuristic, so that reconstruction is fast and informed. Together, these two components give the solver both online adaptivity and offline domain knowledge.

### V.1 Adaptive Operator Selection via Thompson Sampling

The solver faces a recurring decision at every iteration: which of the three destroy operators — Random, Worst-load, or Related-item — should be used? Different operators tend to be more effective at different stages of the search and on different instance structures. Random destruction is useful early on, when the solution still has room for broad exploration; targeted destruction of poorly loaded bins tends to yield better returns once the solution has tightened up. But there is no way to know in advance which operator will be most productive at any given moment — that depends on the instance and the current state of the solution.

The natural response is to *learn from experience*. Each time an operator is used, the outcome — did it lead to a new best solution, or not? — provides information about how valuable that operator currently is. The solver accumulates these signals over the run and uses them to steer selection toward operators that have been consistently rewarding, while still occasionally trying less-used ones that might yet prove useful.

This kind of problem — choosing repeatedly among options with unknown payoffs, trying to balance learning about options you haven't tried much against exploiting the ones that have worked well — is classically called the **multi-armed bandit** problem. The name is a playful analogy: imagine a gambler sitting in front of a row of slot machines (each informally called a *one-armed bandit*), each with a different and unknown payout rate. The gambler must decide, round by round, which machine to play based only on the outcomes of past plays. The core tension is the same as in operator selection: **explore** to gather information about uncertain options, or **exploit** the option currently believed to be best?

The algorithm used here to resolve that tension is called **Thompson Sampling**, named after the statistician William R. Thompson who introduced it in 1933. Its principle is intuitive: for each operator, maintain a running sense of not just how well it has done, but *how confident* you are in that estimate. When an operator has been used rarely, your estimate of its quality is uncertain — it might be much better or much worse than it appears. Thompson Sampling takes that uncertainty seriously: rather than always picking the operator with the best observed track record, it occasionally promotes an uncertain operator to the top just because, given the uncertainty, it *could* be the best. As evidence accumulates and estimates become more reliable, selection naturally concentrates on whichever operator has proven most consistently productive. No manual tuning of exploration rates is required.

**Decision and update cycle.** Concretely, at each iteration:
1. For each operator $k$, draw a single random sample representing a plausible estimate of its current success rate: $\tilde{\theta}_k \sim \mathrm{Beta}(\alpha_k, \beta_k)$.
2. Select the operator with the highest sampled estimate: $k^* = \arg\max_k \tilde{\theta}_k$.
3. Execute operator $k^*$, run repair, and apply the SA acceptance criterion.
4. Observe reward $r = \mathbf{1}[\text{accepted} \cap \text{new best solution found}]$.
5. Update the belief for operator $k^*$: increment $\alpha_{k^*}$ by 1 on success, or $\beta_{k^*}$ by 1 on failure.

The underlying statistical model is a **Beta-Bernoulli** conjugate pair, which affords closed-form updates and requires no numerical integration. All operators start from the same neutral prior $\mathrm{Beta}(1,1)$, meaning no operator is favored at the outset. The formal details of this model are standard and can be found in any treatment of Bayesian bandits.

### V.2 Machine-Learned Repair: Behavioral Cloning from BFD

After destruction, each displaced item $i$ must be reinserted. For item $i$, let $\mathcal{F}(i) = \{j : \text{bin } j \text{ has sufficient residual capacity for item } i\}$ be the set of **feasible bins**. The repair algorithm must select one bin from $\mathcal{F}(i)$ (or open a new one if $\mathcal{F}(i) = \emptyset$).

This is cast as a **binary classification** problem: given a $(\text{item}, \text{bin})$ pair, predict whether this is the *preferred* placement. Items are inserted in non-increasing size order (large items first, mirroring BFD's ordering), and after each placement the bin loads are updated before the next item is considered.

#### V.2.1 Feature representation

Each feasible $(i, j)$ pair is encoded as a 14-dimensional feature vector. Features are normalized by the bin capacity $C$ to ensure scale-independence:

| Index | Feature | Description |
|-------|---------|-------------|
| 0 | $s_i / C$ | Normalized item size |
| 1 | $(s_i / C)^2$ | Squared normalized size (captures non-linearity) |
| 2 | $s_i / C$ | Repeated; redundant but harmless to the linear model |
| 3 | $\mathrm{rank}(i) / n$ | Size rank of item $i$ among all items, normalized |
| 4 | $\text{remaining} / n$ | Fraction of displaced items not yet reinserted |
| 5 | $\ell_j / C$ | Current load of bin $j$, normalized |
| 6 | $(C - \ell_j) / C$ | Residual capacity of bin $j$ |
| 7 | $(C - \ell_j - s_i) / C$ | Slack after placement (post-placement residual) |
| 8 | $(C - \ell_j - s_i) / C$ | Repeated slack (same value) |
| 9 | $\ell_j / C$ | Repeated load |
| 10 | $|B_j| / n$ | Occupancy of bin $j$ (item count, normalized) |
| 11 | $\max_{k \in B_j} s_k / C$ | Largest item currently in bin $j$ |
| 12 | $\min_{k \in B_j} s_k / C$ | Smallest item currently in bin $j$ |
| 13 | $s_i / (C - \ell_j)$ | Fill ratio: how much of the remaining space does item $i$ consume? |

Features 2, 8, and 9 duplicate earlier features; these are artifacts of the feature construction pipeline and do not harm the logistic model (their coefficients can absorb or cancel the duplicated information).

**Feature contract.** The feature vectors produced at training time (`train_repair_model.py::_make_features`) and at inference time (`solver.py::_make_repair_features`) must be identical in structure and semantics. The training script operates on *normalized* sizes in $[0.1, 0.9]$ with capacity 1.0, while the solver works with *integer* sizes and capacity $C$; the normalization in `_make_repair_features` ensures the two domains match.

#### V.2.2 Model architecture: logistic regression

The model is a **logistic regression** trained with the L-BFGS optimizer:

$$
P(\text{preferred} \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b), \qquad \sigma(z) = \frac{1}{1+e^{-z}}.
$$

At repair time, `predict_proba` is called for all feasible bins simultaneously and the bin with the **highest predicted probability** of being preferred is selected. The model is lightweight (14 weights + bias), making inference negligible compared to the combinatorial search overhead.

#### V.2.3 Offline training: imitation learning from BFD

Training data is generated by **replaying BFD decisions** on synthetic instances:
- **Positive example** ($y=1$): the $(\text{item}, \text{bin})$ pair that BFD actually chose (minimum-slack feasible bin).
- **Negative examples** ($y=0$): all other feasible bins at that step, optionally capped at `max_negatives` to control class imbalance.

This is a form of **behavioral cloning** (imitation learning): the model learns to mimic the BFD oracle's placement decisions. BFD is a strong constructive heuristic that tends to leave small residual gaps, and the model internalizes this preference through the slack features (indices 7–9). During ALNS, the model is used as an *efficient approximation* to BFD-style repair, while the bandit-driven destroy phase provides the global diversity that BFD alone cannot achieve.

## VI. Workflow Example

The workflow consists of two steps: training the repair model, then running the benchmark. Each is invoked via a shell command from the notebook.

### VI.1 Step 1 — Training the repair model

**Parameter reference:**

| Flag | Type | Default | Meaning |
|------|------|---------|--------|
| `--instances` | `int` | 5000 | Number of synthetic BPP instances to generate. Each instance contributes a variable number of training rows (one positive + up to `max-negatives` negatives per insertion step where at least two feasible bins exist). With defaults, this typically yields $\sim$500k–1M rows. |
| `--n-min` | `int` | 50 | Minimum number of items in a generated instance. Must be $\ge 1$. |
| `--n-max` | `int` | 200 | Maximum number of items in a generated instance. Must be $\ge$ `n-min`. |
| `--max-negatives` | `int` | 5 | Maximum number of negative examples (non-BFD feasible bins) sampled per insertion step. Caps the class imbalance when many bins are open; 0 disables negatives entirely (not recommended). |
| `--seed` | `int` | 0 | NumPy random seed for reproducibility. Controls both instance generation and negative sampling. |
| `--output` | `str` | `repair_model.pkl` | Output path for the serialized `sklearn` LogisticRegression object. Parent directories are created automatically. |

**What the script does internally:**
1. Generates `instances` synthetic instances with item sizes drawn from $\mathrm{Uniform}(0.1, 0.9)$ and implicit capacity 1.0.
2. Replays BFD on each instance, emitting one training row per feasible multi-bin insertion step.
3. Splits data 80/20 into train/validation sets (stratified on label), fits `LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)`, reports validation accuracy, and serializes the fitted model with `pickle`.

**Scaling advice:**  
- Increasing `--instances` improves generalization at the cost of training time (roughly linear).
- Increasing `--n-max` generates harder instances with more open bins per step, producing richer negative samples but also more rows per instance.
- A `--max-negatives` value between 3 and 10 is a reasonable range; very small values starve the model of negative signal, while very large values slow training without proportionate benefit.

In [1]:
!python train_repair_model.py \
  --instances 5000 \
  --n-min 50 \
  --n-max 200 \
  --max-negatives 5 \
  --seed 0 \
  --output repair_model.pkl

Generated 500/5000 instances...
Generated 1000/5000 instances...
Generated 1500/5000 instances...
Generated 2000/5000 instances...
Generated 2500/5000 instances...
Generated 3000/5000 instances...
Generated 3500/5000 instances...
Generated 4000/5000 instances...
Generated 4500/5000 instances...
Dataset: rows=1286857, cols=14, pos_rate=0.228
Validation accuracy: 0.8060
Saved model to: repair_model.pkl


### VI.2 Step 2 — Running the benchmark

**Parameter reference:**

| Flag | Meaning |
|------|---------|
| `--solver` | Path to the Python module containing `BinPackingSolver`. The benchmark driver imports this module dynamically and instantiates `BinPackingSolver(item_sizes, bin_capacity)`. |
| `--method` | The method name passed to `BinPackingSolver.solve(method=..., **params)`. In this implementation, `method` is accepted but ignored; the solver always follows the hybrid ALNS path. |
| `--method-args` | Comma-separated `key=value` pairs forwarded to `solve()` as keyword arguments. See the table below for recognized keys. |
| `--dataset` | Which benchmark dataset to load. `falkenauer-t` refers to the *triplet* instances of Falkenauer (1996), a standard hard benchmark where items come in groups of three that together fill exactly one bin. |
| `--limit` | Number of benchmark instances to evaluate. Useful for quick experiments; remove or increase for a full benchmark run. |

**Recognized `--method-args` keys:**

| Key | Type | Default | Meaning |
|-----|------|---------|--------|
| `model_path` | `str` | *required* | Path to the serialized repair model produced by `train_repair_model.py`. The solver raises `ValueError` if this is absent. |
| `max_iterations` | `int` | 5000 | Total number of destroy-repair-accept cycles to perform. More iterations improve solution quality at linear cost. |
| `initial_temperature` | `float` | $1/\ln 2 \approx 1.443$ | Starting temperature $T_0$ for the SA acceptance criterion. At $T_0 = 1/\ln 2$, a one-bin degradation is accepted with probability exactly 0.5 at iteration 0. |
| `alpha_cool` | `float` | 0.9995 | Geometric cooling factor $\alpha_{\text{cool}} \in (0, 1]$. Values closer to 1 cool more slowly (more exploration); values closer to 0 cool faster (quicker convergence, risk of premature commitment). |

In [11]:
!python ../../benchmark.py \
  --solver 5_hybrid_ml_metaheuristics/hybrid_alns/solver.py \
  --method hybrid_alns \
  --dataset falkenauer-u \
  --method-args "model_path=repair_model.pkl,max_iterations=5000" \
  --time-limit 50


Starting Benchmark: Falkenauer U
──────────────────────────────────────────────────────────────────────────────────────────────────────
Instance            │ Items │ Capacity │ LB  │ Bins │ Gap  │ Time (s)   │ Method               │ State
──────────────────────────────────────────────────────────────────────────────────────────────────────
Falkenauer_u120_00  │ 120   │ 150      │ 48  │ 48   │ 0    │ 9.3181     │ hybrid_alns          │ Done 
Falkenauer_u120_01  │ 120   │ 150      │ 49  │ 49   │ 0    │ 7.9787     │ hybrid_alns          │ Done 
Falkenauer_u120_02  │ 120   │ 150      │ 46  │ 46   │ 0    │ 8.0763     │ hybrid_alns          │ Done 
Falkenauer_u120_03  │ 120   │ 150      │ 49  │ 49   │ 0    │ 8.0264     │ hybrid_alns          │ Done 
Falkenauer_u120_04  │ 120   │ 150      │ 50  │ 50   │ 0    │ 8.8678     │ hybrid_alns          │ Done 
Falkenauer_u120_05  │ 120   │ 150      │ 48  │ 48   │ 0    │ 9.2074     │ hybrid_alns          │ Done 
Falkenauer_u120_06  │ 120   │ 150      